### Imports

In [1]:
from fundus_dataset import AugmentPair, FundusVesselDataset, CenterCropPair
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset
from monai.networks.nets import UNet
from monai.losses import DiceLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### Testing the Dataset

In [2]:
img_dir = "/workspace/train/Original/"
mask_dir = "/workspace/train/Ground truth"


train_full = FundusVesselDataset(
    img_dir=img_dir,
    mask_dir=mask_dir,
    transform=AugmentPair(crop_size=(512, 512)),
)

val_full = FundusVesselDataset(
    img_dir=img_dir,
    mask_dir=mask_dir,
    transform=None,
)

num_samples = len(train_full)
generator = torch.Generator().manual_seed(42)

# Shuffle the indices
indices = torch.randperm(num_samples, generator=generator).tolist()

train_size = int(0.8 * num_samples)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(val_full, val_indices)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

images, masks = next(iter(train_loader))
val_images, val_masks = next(iter(val_loader))

print("Train images:", images.shape, images.dtype, images.min().item(), images.max().item())
print("Train masks: ", masks.shape, masks.dtype, torch.unique(masks))

print("Val images:", val_images.shape, val_images.dtype, val_images.min().item(), val_images.max().item())
print("Val masks: ", val_masks.shape, val_masks.dtype, torch.unique(val_masks))

Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Train size: 480
Val size: 120
Train images: torch.Size([4, 3, 512, 512]) torch.float32 0.0 0.729411780834198
Train masks:  torch.Size([4, 1, 512, 512]) torch.float32 tensor([0., 1.])
Val images: torch.Size([1, 3, 2048, 2048]) torch.float32 0.0 1.0
Val masks:  torch.Size([1, 1, 2048, 2048]) torch.float32 tensor([0., 1.])


### BCEDice Loss Functions

In [3]:
class BCEDiceLoss(nn.Module):
    def __init__(self, w_bce=0.5, w_dice=0.5):
        super().__init__()
        self.w_bce = w_bce
        self.w_dice = w_dice

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss(sigmoid=True)

    def forward(self, logits, masks):
        bce = self.bce(logits, masks)
        dice = self.dice(logits, masks)

        return self.w_bce * bce + self.w_dice * dice

def hard_dice_score(logits, masks, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    # flatten each image separately: [B, 1, H, W] -> [B, pixels]
    preds = preds.flatten(start_dim=1)
    masks = masks.flatten(start_dim=1)

    intersection = (preds * masks).sum(dim=1)
    denominator = preds.sum(dim=1) + masks.sum(dim=1)

    dice = (2 * intersection + eps) / (denominator + eps)

    return dice.mean()

### U-Net Factory

In [4]:
def make_unet(channels=(16, 32, 64, 128, 256)):
    model = UNet(
        spatial_dims=2,
        in_channels=3,
        out_channels=1,
        channels=channels,
        strides=(2, 2, 2, 2),
        num_res_units=2,
    )
    return model.to(device)

### Training Loop Function

In [5]:
def train_model(model, train_loader, val_loader, loss_fn, optimizer, epochs, save_path=None):
    history = []
    best_val_dice = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = loss_fn(logits, masks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        val_dice = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)

                logits = model(images)
                loss = loss_fn(logits, masks)
                dice = hard_dice_score(logits, masks, threshold=0.5)

                val_loss += loss.item() * images.size(0)
                val_dice += dice.item() * images.size(0)

        val_loss /= len(val_loader.dataset)
        val_dice /= len(val_loader.dataset)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_dice": val_dice,
        })

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_epoch = epoch + 1

            if save_path is not None:
                torch.save(model.state_dict(), save_path)

        print(
            f"Epoch {epoch + 1:03d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f} | "
            f"Val Dice: {val_dice:.4f} | "
            f"Best: {best_val_dice:.4f} @ {best_epoch}"
        )

    return history

### DANGEROUS (RESET EXPERIMENT)

In [6]:
model = make_unet(channels=(32, 64, 128, 256, 512))
loss_fn = BCEDiceLoss(w_bce=0.5, w_dice=0.5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### START OR CONTINUE EXPERIMENT

In [8]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=100,
    save_path="/workspace/models/best_bce_dice_unet_wide.pt",
)

KeyboardInterrupt: 

### Save your Settings

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history)
history_df.to_csv("/workspace/models/history_bce_dice_unet_wide.csv", index=False)

### Scratchpad Code (Don't Bother Deleting anything Below This. It's Junk)

In [ ]:
import json

config = {
    "model": "MONAI UNet",
    "channels": [32, 64, 128, 256, 512],
    "loss": "0.5 BCEWithLogitsLoss + 0.5 DiceLoss(sigmoid=True)",
    "optimizer": "Adam",
    "lr": 1e-3,
    "epochs": 100,
    "crop_size": [512, 512],
    "batch_size": 4,
    "train_val_split": "80/20",
    "seed": 42,
    "validation": "center crop 512x512",
    "best_val_dice": 0.8232,
}

with open("/workspace/models/config_bce_dice_unet_wide_100epochs.json", "w") as f:
    json.dump(config, f, indent=2)

In [ ]:
images, masks = next(iter(val_loader))

print("images:", images.shape)
print("masks: ", masks.shape)

model.eval()

images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    logits = model(images)

print("logits:", logits.shape)

In [ ]:
def validate_full_image_dice(model, loader, threshold=0.5):
    model.eval()
    total_dice = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            dice = hard_dice_score(logits, masks, threshold=threshold)

            total_dice += dice.item() * images.size(0)

    return total_dice / len(loader.dataset)


full_dice = validate_full_image_dice(
    model=model,
    loader=val_loader,
    threshold=0.5,
)

print("Full-image val Dice:", full_dice)

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_val_dice": 0.8916996493935585,
    "model_config": {
        "spatial_dims": 2,
        "in_channels": 3,
        "out_channels": 1,
        "channels": [32, 64, 128, 256, 512],
        "strides": [2, 2, 2, 2],
        "num_res_units": 2,
    },
    "loss": "0.5 * BCEWithLogitsLoss + 0.5 * DiceLoss(sigmoid=True)",
    "optimizer": "Adam",
    "lr": 1e-3,
    "crop_size": [512, 512],
    "batch_size": 4,
    "seed": 42,
    "validation_protocol": "full-image validation",
}

torch.save(checkpoint, "/workspace/models/bce_dice_unet_wide_rich_checkpoint.pt")

In [ ]:
ckpt = torch.load(
    "/workspace/models/4-30-2026-9-49/bce_dice_unet_wide_rich_checkpoint.pt",
    map_location=device,
)

test_model = make_unet(channels=tuple(ckpt["model_config"]["channels"]))
test_model.load_state_dict(ckpt["model_state_dict"])
test_model.eval()

print("Checkpoint reload successful")

full_dice = validate_full_image_dice(
    model=test_model,
    loader=val_loader,
    threshold=0.5,
)

print("Reloaded full-image val Dice:", full_dice)

In [ ]:
notes = """
Baseline completed.

Dataset: FIVES train split only, 80/20 internal split.
Model: MONAI U-Net, channels=(32,64,128,256,512).
Loss: 0.5 BCE + 0.5 Dice.
Training: random 512x512 crops, batch size 4, Adam lr=1e-3, 100 epochs.
Center-crop best val Dice: 0.8232.
Full-image validation Dice: 0.8916996493935585.
Important: center-crop validation underestimated full-image performance.
Next step: compare losses using full-image validation.
"""

with open("/workspace/models/4-30-2026-9-49/README_baseline.txt", "w") as f:
    f.write(notes)

In [ ]:
import os
import torch
import matplotlib.pyplot as plt

save_dir = "/workspace/models/qualitative"
os.makedirs(save_dir, exist_ok=True)

model.eval()

num_examples_to_save = 8
saved = 0

with torch.no_grad():
    for batch_idx, (images, masks) in enumerate(val_loader):
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        batch_size = images.size(0)

        for idx in range(batch_size):
            if saved >= num_examples_to_save:
                break

            fig = plt.figure(figsize=(12, 4))

            plt.subplot(1, 4, 1)
            plt.imshow(images[idx].cpu().permute(1, 2, 0))
            plt.title("Image")
            plt.axis("off")

            plt.subplot(1, 4, 2)
            plt.imshow(masks[idx, 0].cpu(), cmap="gray")
            plt.title("Ground truth")
            plt.axis("off")

            plt.subplot(1, 4, 3)
            plt.imshow(probs[idx, 0].cpu(), cmap="gray", vmin=0, vmax=1)
            plt.title("Probability")
            plt.axis("off")

            plt.subplot(1, 4, 4)
            plt.imshow(preds[idx, 0].cpu(), cmap="gray", vmin=0, vmax=1)
            plt.title("Prediction")
            plt.axis("off")

            plt.tight_layout()

            out_path = os.path.join(
                save_dir,
                f"bce_dice_val_example_{saved:02d}.png"
            )

            plt.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)

            print("Saved:", out_path)

            saved += 1

        if saved >= num_examples_to_save:
            break

print(f"Saved {saved} qualitative examples to {save_dir}")